# [실습 1] Pipeline API로 작은 LLM 추론하기


## 실습 개요

> "작은 LLM을 Pipeline API로 빠르게 실행하면 어떤 흐름으로 추론이 이루어질까?"

이번 실습에서는 Hugging Face Transformers의 고수준 추론 인터페이스인 `pipeline()`을 사용합니다.  
Pipeline은 모델 선택, 토크나이저 로드, 입력 전처리, 모델 추론, 결과 후처리를 하나의 호출 흐름으로 연결합니다.

실습 모델은 앞서 선택한 경량 Instruction LLM인 `HuggingFaceTB/SmolLM2-135M-Instruct`입니다.  
이 모델은 약 135M 규모라 실습 환경에서 Pipeline, 채팅 메시지 입력, 생성 파라미터를 다루기에 적합합니다.


## 실습 목표

1. `pipeline(task, model, **kwargs)` 기본 문법을 이해한다.
2. 작은 LLM을 `text-generation` Pipeline으로 로드한다.
3. 문자열 프롬프트와 채팅 메시지 입력의 차이를 확인한다.
4. `max_new_tokens`, `do_sample`, `temperature`, `top_k` 같은 생성 파라미터를 조절한다.
5. [TODO] 여러 프롬프트의 생성 결과를 표 형태로 정리한다.


## 데이터 출처

- 데이터셋: 실습용 프롬프트 직접 구성
- 모델: `HuggingFaceTB/SmolLM2-135M-Instruct` (Hugging Face Hub)


## 목차

1. 라이브러리 임포트 및 환경 설정
2. Text-generation Pipeline 생성
3. 단일 프롬프트 추론
4. 채팅 메시지 입력 추론
5. 생성 파라미터 비교
6. [TODO] 여러 프롬프트 결과 정리


---
## 1. 라이브러리 임포트 및 환경 설정


In [ ]:
%pip install pandas
%pip install transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 11.4 MB/s  0:00:00 eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [3]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from transformers import pipeline

pd.set_option('display.max_colwidth', 160)
print('준비 완료')

준비 완료


---
## 2. Text-generation Pipeline 생성

`pipeline()`에 `text-generation` 태스크와 모델 ID를 전달하면 생성용 Pipeline이 만들어집니다.  
작은 모델이라도 최초 실행 시에는 모델 파일 다운로드 시간이 걸릴 수 있습니다.


In [4]:
model_name = 'HuggingFaceTB/SmolLM2-135M-Instruct'

generator = pipeline(
    'text-generation',
    model=model_name,
    tokenizer=model_name,
)

print(generator)
print('모델 로드 완료:', model_name)

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 5355.38it/s]


TextGenerationPipeline: {'model': 'LlamaForCausalLM', 'dtype': 'bfloat16', 'device': 'mps', 'input_modalities': 'text', 'output_modalities': ('text',)}
모델 로드 완료: HuggingFaceTB/SmolLM2-135M-Instruct


---
## 3. 단일 프롬프트 추론

Pipeline 객체는 함수처럼 호출할 수 있습니다.  
`max_new_tokens`는 입력 뒤에 새로 생성할 토큰 수를 제한합니다.


In [5]:
prompt = 'Explain why the Hugging Face pipeline API is useful in one sentence.'

result = generator(prompt, max_new_tokens=40, do_sample=False)
print(type(result))
print(result[0]['generated_text'])

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


<class 'list'>
Explain why the Hugging Face pipeline API is useful in one sentence.


---
## 4. 채팅 메시지 입력 추론

Instruction 모델은 `role`과 `content`로 구성된 메시지 리스트를 사용할 수 있습니다.  
Pipeline에 메시지를 전달하면 내부적으로 모델의 채팅 템플릿이 적용됩니다.


In [6]:
messages = [
    {'role': 'system', 'content': 'You are a concise machine learning tutor.'},
    {'role': 'user', 'content': 'What does a tokenizer do in Transformers?'},
]

chat_result = generator(messages, max_new_tokens=50, do_sample=False)
print(chat_result[0]['generated_text'])

[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'role': 'system', 'content': 'You are a concise machine learning tutor.'}, {'role': 'user', 'content': 'What does a tokenizer do in Transformers?'}, {'role': 'assistant', 'content': 'In the Transformers, a tokenizer is a crucial component that breaks down the text into individual words, words into words, and words into words into words. It\'s responsible for identifying and removing stop words, such as "the," "and," and'}]


In [7]:
print(chat_result[0]['generated_text'][-1]['content'])

In the Transformers, a tokenizer is a crucial component that breaks down the text into individual words, words into words, and words into words into words. It's responsible for identifying and removing stop words, such as "the," "and," and


---
## 5. 생성 파라미터 비교

Greedy decoding은 가장 높은 점수의 토큰을 반복적으로 선택합니다.  
Sampling은 확률적으로 토큰을 선택하므로 `temperature`, `top_k`에 따라 결과가 달라질 수 있습니다.


In [8]:
sample_prompt = 'A practical tip for using Transformers is'

greedy = generator(sample_prompt, max_new_tokens=30, do_sample=False)[0]['generated_text']
sampled = generator(
    sample_prompt,
    max_new_tokens=30,
    do_sample=True,
    temperature=0.8,
    top_k=20,
)[0]['generated_text']

print('[Greedy]')
print(greedy)
print('\n[Sampling]')
print(sampled)

[transformers] Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'temperature', 'top_k'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[Greedy]
A practical tip for using Transformers is to consider the following:

* Use them to create a narrative that's engaging and relatable. Transformers are often used to explore complex themes and

[Sampling]
A practical tip for using Transformers is to be prepared to adapt your story as needed. Your characters will need to adjust to new information, relationships, and situations, so be ready to make


---
### [TODO] 여러 프롬프트 결과를 표로 정리하기

각 프롬프트에 대해 생성 결과를 만들고, 프롬프트 원문과 생성 텍스트, 새로 생성된 문자 수를 표로 정리합니다.

반복문은 `rows` 리스트에 `prompt`, `generated_text`, `new_text_length` 값을 가진 딕셔너리를 차례로 추가합니다. 마지막 줄에서 `rows`를 `pd.DataFrame(rows)`로 변환해 `generation_df`에 저장하세요.


In [ ]:
practice_prompts = [
    'Transformers pipeline is very helpful because',
    'AutoModel classes are useful when',
    'KV cache improves generation by',
]

rows = []
for prompt in practice_prompts:

    result = generator(prompt, max_new_tokens=25, do_sample=False)
    generated = result[0]['generated_text']
    
    rows.append({
        'prompt': prompt,
        'generated_text': generated,
        'new_text_length': len(generated) - len(prompt),
    })

generation_df = pd.DataFrame(data=rows)

generation_df


[transformers] Both `max_new_tokens` (=25) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=25) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=25) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


,prompt,generated_text,new_text_length
0,Transformers pipeline is very helpful because,"Transformers pipeline is very helpful because it allows you to:\n\n1. **Analyze and visualize data**: You can analyze the data to identify trends, patterns",108
1,AutoModel classes are useful when,"AutoModel classes are useful when you need to create a model that can be used for multiple purposes, such as classification, regression, or clustering.\n\n",120
2,KV cache improves generation by,"KV cache improves generation by 1.5%\n\nThe results are not drastically different from the original data, but they are more consistent and reliable",114


In [13]:
print(generation_df)
print('행 개수:', len(generation_df), '/', len(practice_prompts))
print('생성 길이 최솟값:', generation_df['new_text_length'].min())

                                          prompt  \
0  Transformers pipeline is very helpful because   
1              AutoModel classes are useful when   
2                KV cache improves generation by   

                                                                                                                                                generated_text  \
0  Transformers pipeline is very helpful because it allows you to:\n\n1. **Analyze and visualize data**: You can analyze the data to identify trends, patterns   
1  AutoModel classes are useful when you need to create a model that can be used for multiple purposes, such as classification, regression, or clustering.\n\n   
2          KV cache improves generation by 1.5%\n\nThe results are not drastically different from the original data, but they are more consistent and reliable   

   new_text_length  
0              108  
1              120  
2              114  
행 개수: 3 / 3
생성 길이 최솟값: 108
